In [77]:
# Problem 1
import pandas as pd
import numpy as np


df = pd.read_stata('A1_kommune.dta')

vars = ['taxrev', 'taxrate', 'pop']
desc = df[vars].describe()

print(desc.round(2))

         taxrev  taxrate        pop
count     98.00    98.00      98.00
mean    4477.34    25.21   56475.89
std     5251.18     0.91   62925.30
min      211.23    22.80    1969.00
25%     2466.70    24.80   29997.75
50%     3317.85    25.30   43475.00
75%     4786.06    25.70   59733.00
max    44170.34    27.80  528208.00


In [100]:
#2.3
#Tilføje log-parameter + konstantsøjle
df['konstant']=1
df['log_taxrev'] = np.log(df['taxrev'])

#Lave input
X23 = df[['konstant', 'taxrate']].values
y = df['log_taxrev'].values

#OLS funktion (fra github øvelse)
def OLS(X, y):
    n = X.shape[0]
    k = X.shape[1] - 1

    betahat = np.linalg.inv(X.T @ X) @ X.T @ y
    yhat = X @ betahat
    uhat = y - X @ betahat
    
    ybar = y.mean()

    SST = np.sum((y - ybar)**2)
    SSE = np.sum((yhat - ybar)**2)
    SSR = np.sum(uhat**2)

    R2 = SSE / SST
    
    sigma2 = np.dot(uhat, uhat) * 1/(n-k-1)

    var_betahat = sigma2 * np.linalg.inv(X.T @ X)
    se_betahat = np.sqrt(np.diag(var_betahat))

    return {
        'betahat': betahat,
        'se_betahat': se_betahat,
        'yhat': yhat,
        'uhat': uhat,
        'sigma2': sigma2,
        'R2': R2,
        'SST': SST,
        'SSE': SSE,
        'SSR': SSR,
    }
    
#Kalder OLS på datasættet
results23 = OLS(X23, y)

#Tabeller og print funktioner
table23 = pd.DataFrame({
    'Delta hat': results23['betahat'],
    'Std. Error': results23['se_betahat']
    }, index=['Constant', 'Taxrate'])


print(table23.round(4))
print('R2 =',results23['R2'].round(4))

          Delta hat  Std. Error
Constant    11.6982       2.143
Taxrate     -0.1426       0.085
R2 = 0.0285


In [104]:
# Problem 2.5

#Tilføjer log pop parameter
df['log_pop'] = np.log(df['pop'])


#Lav ny X input
X25 = df[['konstant', 'taxrate', 'log_pop']].values

#Kalder OLS på datasættet
results25 = OLS(X25, y)

#Tabeller og print funktioner
table25 = pd.DataFrame({
    'Beta hat': results25['betahat'],
    'Std. Error': results25['se_betahat']
    }, index=['Constant', 'Taxrate', 'Log Population'])

print(table25.round(4))
print('R2 =',results25['R2'].round(4))

                Beta hat  Std. Error
Constant         -2.8022      0.3756
Taxrate           0.0226      0.0125
Log Population    0.9711      0.0144
R2 = 0.9801


In [129]:
# Problem 3.3

#Første regression 
#Nye input
y1 = df['taxrate'].values
X1 = df[['konstant','log_pop']].values

results1 = OLS(X1, y1)
#Gemme residualer

df['res1'] = results1['uhat']

#Anden regression
y2 = df['log_taxrev'].values
X2 = df[['konstant', 'res1']].values

results2 = OLS(X2, y2)

#Gemme residualer
print("beta_1 hat =", results2['betahat'][1])

beta_1 hat = 0.022622264058285026
